In [1]:
!pip install pdfplumber faiss-cpu numpy sentence-transformers together

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 732.0 kB/s eta 0:00:000:01
  Using cached pypdfium2-4.30.0-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (48 kB)
  Using cached charset_normalizer-3.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (33 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
  Using cached tqdm-4.66.5-py3-none-any.whl.metadata (57 kB)
  Using cached torch-2.4.1-cp312-cp312-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached scipy-1.14.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached huggingface_hub-0.24.7-py3-none-any.whl.metadata (13 kB)
  Using cached click-8.1.7-py3-none-any.whl.metadata (3.0 kB)
  Using cached filelock-3.16.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached pydantic-2.9.1-py3-none-any.whl.metadata (146 kB)
  

In [ ]:
raw_data_folder = "../data/raw"
processed_data_folder = "../data/processed"

In [33]:
import pdfplumber
import faiss
import numpy as np
import json
from sentence_transformers import SentenceTransformer
from together import Together
from docx import Document

In [32]:
!pip install python-docx 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 817.7 kB/s eta 0:00:00a 0:00:01


In [34]:
# Initialize Together AI client
together_client = Together(api_key='46d9e04509efe0eb914936cc15d795f2104779b993b0bbc1b821fa5c49b20567')

# Initialize embedding model
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')  # Adjust model as needed

def create_embedding(text):
    embedding = embedding_model.encode(text, convert_to_numpy=True)
    return embedding

def read_docx_file(file_path):
    document = Document(file_path)
    chunks = []
    section = None
    subsection = None

    for para in document.paragraphs:
        # Check if the paragraph is a heading
        if para.style.name.startswith('Heading'):
            # Determine heading level
            heading_level = para.style.name.replace('Heading', '').strip()
            if heading_level.isdigit():
                heading_level = int(heading_level)
                if heading_level == 1:
                    section = para.text.strip()
                    subsection = None  # Reset subsection when a new section starts
                elif heading_level == 2:
                    subsection = para.text.strip()
                # Add more levels if needed
        else:
            text = para.text.strip()
            if text:
                chunk = {
                    'id': len(chunks),
                    'text': text,
                    'page': None,  # Pages are not applicable for DOCX
                    'section': section,
                    'subsection': subsection
                }
                chunks.append(chunk)
    return chunks


def build_recursive_json(chunks, hierarchy_keys):
    tree = {}
    for chunk in chunks:
        current_level = tree
        for key in hierarchy_keys:
            key_value = chunk.get(key)
            if key_value:
                if key_value not in current_level:
                    current_level[key_value] = {}
                current_level = current_level[key_value]
        if 'texts' not in current_level:
            current_level['texts'] = []
        current_level['texts'].append({
            'id': chunk['id'],
            'text': chunk['text']
        })
    return tree


def batch_chunks(chunks, max_chars=2000):
    batches = []
    current_batch = []
    current_chars = 0

    for chunk in chunks:
        chunk_chars = len(chunk['text'])
        if current_chars + chunk_chars > max_chars:
            batches.append(current_batch)
            current_batch = []
            current_chars = 0
        current_batch.append(chunk)
        current_chars += chunk_chars

    if current_batch:
        batches.append(current_batch)
    return batches

def build_summary_json(summarized_chunks):
    json_list = []
    for chunk in summarized_chunks:
        chunk_json = {
            'id': chunk['id'],
            'text': chunk['text'],
            'level': chunk['level'],
            'children': []
        }
        if 'children' in chunk:
            chunk_json['children'] = build_summary_json(chunk['children'])
        json_list.append(chunk_json)
    return json_list

/home/malek/AllamChallenge/venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [70]:
def summarize_with_llama(text):
    # Use Together AI's LLaMA model for summarization

    text = str(text)

    prompt=f"please extract the arabic potery excerpts and it's analysis from :\n\n{str(text)}\n\:",

    print(prompt)

    messages=[
        {
                "role": "user",
                "content": prompt
        }
    ]


    response = together_client.chat.completions.create(
        model="meta-llama/Meta-Llama-3-8B-Instruct-Lite",
        messages=messages,
        temperature=0.7,
        repetition_penalty=1,
        stop=["<|eot_id|>","<|eom_id|>"],
    )

    summary = response.get('output', '')
    return summary.strip()


def recursive_summarization(chunks, level=0, max_level=3):
    # Base case: if only one chunk or max level reached
    if len(chunks) == 1 or level >= max_level:
        return chunks

    batches = batch_chunks(chunks)
    summarized_chunks = []
    for batch in batches:
        combined_text = "\n".join([chunk['text'] for chunk in batch])
        
        print(combined_text)
        
        summary_text = summarize_with_llama(combined_text)
        summarized_chunk = {
            'id': f'summary_level_{level}_{len(summarized_chunks)}',
            'text': summary_text,
            'level': level,
            'children': batch
        }
        summarized_chunks.append(summarized_chunk)

    # Recursive call
    return recursive_summarization(summarized_chunks, level + 1, max_level)

<>:4: SyntaxWarning: invalid escape sequence '\:'
<>:4: SyntaxWarning: invalid escape sequence '\:'
/tmp/ipykernel_70576/939991828.py:4: SyntaxWarning: invalid escape sequence '\:'
  prompt=f"please extract the arabic potery excerpts and it's analysis from :\n\n{str(text)}\n\:",


In [71]:
relevant_chunks = []

def main():
    # Step 1: Extract text and create chunks with metadata
    chunks = []
    with pdfplumber.open('/home/malek/AllamChallenge/data/raw/التألق 21 نوفمبر-50-100.pdf') as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()
            if text:
                sections = split_into_sections(text)
                for section in sections:
                    chunk = {
                        'id': len(chunks),
                        'text': section['text'],
                        'page': page_num,
                        'section': section.get('heading'),
                        'subsection': section.get('subheading')
                    }
                    chunks.append(chunk)

    # Step 2: Create embeddings and build the index
    texts = [chunk['text'] for chunk in chunks]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    # Step 3: Retrieve relevant chunks based on a query
    query = "مقتطفات من الشعر العربي ومعانيها" # Your question in Arabic
    query_embedding = create_embedding(query)   
    k = len(chunks)  # Retrieve all chunks
    distances, indices = index.search(np.array([query_embedding]), k)

    # Define a relevance threshold
    relevance_threshold = np.percentile(distances[0], 10)  # Top 10% most similar
    relevant_indices = [idx for dist, idx in zip(distances[0], indices[0]) if dist <= relevance_threshold]
    relevant_chunks = [chunks[i] for i in relevant_indices]

    # Step 4: Build recursive JSON output
    if relevant_chunks:
        print(relevant_chunks[0])
        #summarized_chunks = recursive_summarization(relevant_chunks)
        #json_output = build_summary_json(summarized_chunks)

        pass

        # Step 5: Save JSON outputs
"""         with open('rag_output.json', 'w', encoding='utf-8') as f:
            json.dump(json_output, f, ensure_ascii=False, indent=4)
    else:
        print("No relevant chunks found.") """

if __name__ == '__main__':
    main()

.تارا(cid:7)تخم نم رث(cid:7)كأ ت(cid:7)سيل يه .ا(cid:7)هل سيل ا(cid:7)م ةل(cid:7)سلسلا هذهل معزأ لا
لاو مهف(cid:7)صن لاو ءارع(cid:7)شلا لك مضت نلف .كلذك يهف اهئارعش يف ىتحو
رع(cid:7)(cid:7)شلا را(cid:7)(cid:7)تخأ ا(cid:7)(cid:7)نأف .مهزر(cid:7)(cid:7)بأ مهنأ دU (cid:7)(cid:7)قتعأ نم ط(cid:7)(cid:7)قف م(cid:7)(cid:7)ضتس .مهر(cid:7)(cid:7)شع
\
.اضيأ ءارعشلا راتخأ يقوذبو ؛يقوذب
\ \
سيل ا(cid:7)(cid:7)نورق وأ ا(cid:7)(cid:7)نرق ل(cid:7)(cid:7)مهأ د(cid:7)(cid:7)ق .ه(cid:7)(cid:7)ب يل نأش لاف يخيراتلا ليثمتلا امأو
ا(cid:7)(cid:7)م لك .ثداوحلا رابكل لا ءارعشلا رابكل ةلسلس هذه .ميظع رعاش اهيف
ا(cid:7)(cid:7)م قيقد(cid:7)(cid:7)ت يف يwنيع رو(cid:7)(cid:7)ن نم كا(cid:7)(cid:7)نيع ه(cid:7)(cid:7)ب (cid:149)ر(cid:7)(cid:7)قw wت ا(cid:7)(cid:7)م ق(cid:7)(cid:7)فنأ نأ كل هنمضأ
o
\
.كسفن هيلإ نئمطت احيحص يتأي ىتح هليكشتو كل هراتخأ
1431 ةجحلا وذ 25 - 2010 لولأا نوناك /ربمسيد 2 ةحودلا
ةيميمح 1
حU يرلاو ،انعU ج
t
اضم هt يف ت(cid:134) دw wرwب اذإ ل
(cid:145)
وليأ لU يل اذ oبح اي
ءU اوج(cid:134) سw
ةنكاس

In [72]:
def main():
    # Step 1: Read and extract data from DOCX files
    file_path = '/home/malek/AllamChallenge/data/raw/part-1.docx'  # Replace with your DOCX file path
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    chunks = read_docx_file(file_path)
    if not chunks:
        print("No text found in the DOCX file.")
        return

    # Step 2: Create embeddings and build the vector index
    texts = [chunk['text'] for chunk in chunks]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    # Step 3: Retrieve relevant chunks based on a query
    query = "سؤالك هنا"  # Your question in Arabic
    query_embedding = create_embedding(query)
    k = len(chunks)  # Retrieve all chunks
    distances, indices = index.search(np.array([query_embedding]), k)

    # Define a relevance threshold
    relevance_threshold = np.percentile(distances[0], 10)  # Top 10% most similar
    relevant_indices = [idx for dist, idx in zip(distances[0], indices[0]) if dist <= relevance_threshold]
    relevant_chunks = [chunks[i] for i in relevant_indices]

    # Step 4: Summarize chunks using Together AI's LLaMA batch API
    if relevant_chunks:
        summarized_chunks = recursive_summarization(relevant_chunks)
        # Step 5: Build a recursive JSON output
        json_output = build_summary_json(summarized_chunks)

        # Step 6: Save the output to a JSON file
        with open('rag_output.json', 'w', encoding='utf-8') as f:
            json.dump(json_output, f, ensure_ascii=False, indent=4)
        print("RAG output saved to rag_output.json")
    else:
        print("No relevant chunks found.")

if __name__ == '__main__':
    main()

ساوروه: صارعوه لامتلاكه
هذه المختارات
هذه السلسلة
ساءَها أنْ رأتْ حَبيباً إليْها 	     ضَاحِكَ الرَّأْسِ عن مَفَارِقَ شِيبِ
هتماً لفيك: فلتتكسر أسنانك
فاسْقِنا مِنْ شرابِكَ الرَّائقِ العَذْ       بِ، ولا تَحْمِنا، سَقَتْكَ السَّماءُ!
بَلْهَ المعابة (ناهيك من معابتهم)
عصر المتنبي.. من ابن الرومي حتى سقوط بغداد
يا مُنْكِرَ المجدِ فِيهِمْ	        أليسَ مِنْهُمْ صُهَيْبُ؟
العوسج: نبت شوكي
فرعك الغربيب: شَعرك الأسود
وقال يهجو شاعراً:
كلام آخر
الحيا: المطر
ابن الرومي
ابن الرومي
شرخ الشباب: أوله
وتُقِرُّ أنَّكَ جَاهِلٌ 		          لم تَأْتِ منْ أَمْرٍ صَوَابَهْ
هُوِيُّ: سقوط (من هَوَى)، السباسب: الصحارى
لا سِيَّمَا بِفَمٍ يَظَلُّ - 		       مَنِيُّ باكَتِهِ شَرَابَهْ
وما التَّبْريكُ في شهرٍ طويلٍ               يُطَاوِلُ يومُهُ يَوْمَ الحِسابِ
الخطة: الأمر
كلُّ شيءٍ أراهُ منكَ بَشيرٌ	           صَدَّقَ اللهُ هذهِ البُشَرَاءَ
قد تُحْسِنُ الرومُ شِعراً	        ما أَحْسَنَتْهُ العُرَيْبُ
فاسْقِنيِ عِشْرينَ رَطلاً		 لا تَشُبْهُنَّ بِمَاءِ
طالَ رَفْوِيِ له فَأَوْدَى بِكَسْبيِ       يا ابنَ حربٍ ترك

/tmp/ipykernel_70576/939991828.py:4: SyntaxWarning: invalid escape sequence '\:'
  prompt=f"please extract the arabic potery excerpts and it's analysis from :\n\n{str(text)}\n\:",


ValidationError: 1 validation error for ChatCompletionRequest
messages.0.content
  Input should be a valid string [type=string_type, input_value=("please extract the arab...لغَنَاءِ\n\\:",), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.9/v/string_type

In [39]:
import os

In [58]:
print(relevant_chunks[0]["text"])

IndexError: list index out of range